# Agent Communication Matrix - Setup & Data Loading

This notebook initializes the demo environment and loads the sample corpus.

In [20]:
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Add project root to path
project_root = Path().cwd().parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Environment loaded from .env")

Project root: /Users/jeremy/Repos/agent-communication-matrix
Environment loaded from .env


## Step 1: Initialize Oracle Connection

In [ ]:
import oracledb

# Get connection parameters from environment
db_user = os.getenv('DB_USER', 'agentuser')
db_password = os.getenv('DB_PASS', 'AgentPass_123')
db_dsn = os.getenv('DB_DSN', 'localhost:1521/FREEPDB1')

print(f"Connecting to Oracle at {db_dsn}...")

try:
    conn = oracledb.connect(
        user=db_user,
        password=db_password,
        dsn=db_dsn
    )
    print("✓ Successfully connected to Oracle Database")
    
    # Verify connection with a simple query
    with conn.cursor() as cursor:
        cursor.execute("SELECT 'Connected!' FROM DUAL")
        result = cursor.fetchone()[0]
        print(f"  Status: {result}")
except Exception as e:
    print(f"✗ Connection failed: {e}")
    conn = None

Connecting to Oracle at localhost:1521/FREEPDB1...
✓ Successfully connected to Oracle Database
✗ Connection failed: ORA-00942: table or view "SYS"."V_$DATABASE" does not exist
Help: https://docs.oracle.com/error-help/db/ora-00942/


## Step 2: Check Vector Table Schema

In [21]:
if conn:
    with conn.cursor() as cursor:
        # Check vector knowledge base table
        cursor.execute("""
            SELECT table_name FROM user_tables WHERE table_name = 'KB_CHUNKS'
        """)
        tables = cursor.fetchall()
        print(f"Vector tables found: {[t[0] for t in tables]}")
        
        if tables:
            # Get schema info
            table_name = 'KB_CHUNKS'
            cursor.execute(f"""
                SELECT column_name, data_type, nullable 
                FROM user_tab_columns 
                WHERE table_name = '{table_name}'
                ORDER BY column_id
            """)
            schema = cursor.fetchall()
            print(f"\nSchema for {table_name}:")
            for col_name, data_type, nullable in schema:
                print(f"  {col_name:20} {data_type:30} {'NOT NULL' if nullable == 'N' else 'NULL'}")
            
            # Row count
            cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
            count = cursor.fetchone()[0]
            print(f"\nTotal records: {count:,}")

Vector tables found: ['KB_CHUNKS']

Schema for KB_CHUNKS:
  ID                   NUMBER                         NOT NULL
  DOC_ID               VARCHAR2                       NOT NULL
  CHUNK_TEXT           CLOB                           NOT NULL
  EMBEDDING            VECTOR                         NULL
  CREATED_AT           TIMESTAMP(6)                   NULL

Total records: 0


## Step 3: Load Sample Data

In [27]:
# Load sample chunks from file
chunks_file = project_root / 'data' / 'chunks.txt'

if chunks_file.exists():
    with open(chunks_file, 'r') as f:
        chunks = f.read().strip().split('\n\n')
    
    print(f"Loaded {len(chunks)} chunks from {chunks_file.name}")
    print(f"\nFirst chunk (preview):")
    print(f"  Length: {len(chunks[0])} characters")
    print(f"  Content: {chunks[0][:200]}...")
else:
    print(f"✗ File not found: {chunks_file}")
    chunks = []

Loaded 1 chunks from chunks.txt

First chunk (preview):
  Length: 5570718 characters
  Content: When working with vector search, engineers must consider fan-out and back-pressure. The relationship between agent memory and availability affects consistency significantly. Production deployments of ...


## Step 4: Verify Sample Records

In [29]:
if conn:
    with conn.cursor() as cursor:
        # Fetch sample records from knowledge base
        cursor.execute("""
            SELECT id, doc_id, chunk_text
            FROM kb_chunks 
            WHERE ROWNUM <= 5
            ORDER BY id
        """)
        samples = cursor.fetchall()
        
        if samples:
            df = pd.DataFrame(
                samples,
                columns=['ID', 'Doc ID', 'Text Preview']
            )
            
            # Truncate text for display
            df['Text Preview'] = df['Text Preview'].astype(str).str[:80] + '...'
            display(df)
        else:
            print("No records found in kb_chunks table")

,ID,Doc ID,Text Preview
0,1,doc_0,"When working with vector search, engineers mus..."
1,2,doc_0,Understanding database transactions requires d...
2,3,doc_0,Modern systems implementing natural language p...
3,4,doc_0,Production deployments of tool calling need ca...
4,5,doc_0,"When working with agent memory, engineers must..."


## Status Check

In [30]:
print("\n" + "="*50)
print("Setup Status")
print("="*50)
print(f"Database connected: {'✓' if conn else '✗'}")
print(f"Chunks loaded: {'✓' if chunks else '✗'}")
print(f"Sample data available: {'✓' if conn and len(samples) > 0 else '✗'}")
print("\nReady to proceed to next notebook!")


Setup Status
Database connected: ✓
Chunks loaded: ✓
Sample data available: ✓

Ready to proceed to next notebook!
